## Import libraries

In [ ]:
import pandas as pd
import os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(override=True)

## Define directories

In [ ]:
notebook_dir = os.getcwd() 
project_root = os.path.dirname(notebook_dir)  
output_dir = os.path.join(project_root, 'data', 'processed') 
os.makedirs(output_dir, exist_ok=True)

## Load Data
### Tasks
- Load the patient, admissions, and icustays data from the CSV file specified in the environment variable.
- Display the columns of the Dataframes to verify that the data has been loaded correctly
- Display the shape of the Dataframes to see the number of rows and columns

In [ ]:
pat_data_path = os.getenv('PAT_DATA_PATH')
df_pat = pd.read_csv(pat_data_path)

In [ ]:
df_pat.columns

In [ ]:
df_pat.shape

In [ ]:
adm_data_path = os.getenv('ADM_DATA_PATH')
df_adm = pd.read_csv(adm_data_path)

In [ ]:
df_adm.columns

In [ ]:
df_adm.shape

In [ ]:
icustays_data_path = os.getenv('ICUSTAYS_DATA_PATH')
df_icustays = pd.read_csv(icustays_data_path)

In [ ]:
df_icustays.columns

In [ ]:
df_icustays.shape

## Stage 1: Build the Base ICU Stay Feature Table
### Tasks
- Create subsets for patients, admissions, and icustays extracting only the columns needed.
    Since we are comparing ICU patients, the ICU stay (stay_id) is chosen as the unit of analysis.
    Each row in the resulting dataset represents one ICU stat and serves as the foundation for all subsequent feature engineering.
- Merge subsets.
    Left join patients on subject_id then left join admissions on hadm_id. Left join ensures every ICU stay remains in the dataset.
- Update column names.
- Save stage 1 CSV into data/processed.

In [ ]:
patients_subset = df_pat[['subject_id', 'gender', 'anchor_age']]
admissions_subset = df_adm[['hadm_id','admission_type', 'race', 'hospital_expire_flag']]
icustays_subset = df_icustays[['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'los']]
 
df_stage1= pd.merge(icustays_subset, patients_subset, on='subject_id', how='left')
df_stage1 = pd.merge(df_stage1, admissions_subset, on='hadm_id', how='left')
df_stage1 = df_stage1.rename(columns={'anchor_age': 'age', 'los': 'icu_los_days'})

In [ ]:
output_file = os.path.join(output_dir, 'icu_stay_features_stage1.csv')
df_stage1.to_csv(output_file, index=False)
print(f"Saved directly to absolute root path:\n{output_file}")

## Stage 2: Diagosis Feature

### Tasks
- Build a reusable pipeline that can be easily extended later.
- Load and merge the diagnoses_icd and d_icd_diagnoses tables.
- Transform diagnosis records into binary clinical features. Does the diagnosis title contain one of the keywords for this condition?
- Aggregate by hospital admission. One admission has many diagnoses. We want One admission -> One Row.
- Merge stage 2 features with stage 1 features.


The original *diagnoses_icd* table contains over **6.3 million** diagnosis records across all hospital admissions in MIMIC-IV. Since this project focuses only on ICU patients, processing every diagnosis record would introduce unnecessary computation.
To improve efficiency, the diagnosis data is filtered in two stages:
1. **Filter to ICU admissions** by retaining only hospital admissions (hadm_id) present in the stage 1 ICU feature table.
2. **Filter to clinically relevant diagnoses** using keyword matching for the selected medical conditions (e.g., hypertension, diabetes, heart failure, chronic kidney disease).

This reduces the dataset from **6,364,488** diagnosis record to **325,269** relevant records-a reduction of approximately **95%**. Performing feature engineering on this smaller subset significantly decreases runtime while preserving the clinical information needed for the patient similarity model.

In [ ]:
# Define a dictionary of keywords for each condition
condition_keywords = {
    "hypertension": [
        "hypertension"
    ],

    "hyperlipidemia": [
        "hyperlipidemia"
    ],

    "diabetes": [
        "diabetes mellitus",
        "type ii diabetes",
        "type 2 diabetes"
    ],

    "heart_failure": [
        "heart failure",
        "congestive heart failure"
    ],

    "coronary_artery_disease": [
        "coronary artery",
        "atherosclerotic heart disease"
    ],

    "atrial_fibrillation": [
        "atrial fibrillation"
    ],

    "chronic_kidney_disease": [
        "chronic kidney disease"
    ],

    "acute_kidney_injury": [
        "acute kidney failure"
    ],

    "anemia": [
        "anemia"
    ],

    "obesity": [
        "obesity"
    ],

    "gerd": [
        "reflux",
        "gastro-esophageal reflux"
    ],

    "uti": [
        "urinary tract infection"
    ]
}

In [ ]:
diag_data_path = os.getenv('DIAG_DATA_PATH')
df_diag = pd.read_csv(diag_data_path)

In [ ]:
diagnoses_data_path = os.getenv('DIAG_CODES_DATA_PATH')
df_diagnoses = pd.read_csv(diagnoses_data_path)

In [ ]:
# merge the diagnoses_icd + d_icd_diagnoses using icd_code and ICD_VERSION to get the full description of the diagnosis codes
df_diag_named = df_diag.merge(
    df_diagnoses,
    on=['icd_code', 'icd_version'],
    how='left'
)

In [ ]:
# Hospital admissions that are in the ICU stays dataset
icu_hadm_ids = df_stage1['hadm_id'].unique()

# Keep only diagnosis for those admissions
df_diag_icu = df_diag_named[
    df_diag_named['hadm_id'].isin(icu_hadm_ids)
].copy()

print(f"Original diagnoses: {len(df_diag_named):,}") 
print(f"ICU diagnoses: {len(df_diag_icu):,}") 

In [ ]:
# apply lowercase to the description column for easier matching
df_diag_icu['long_title'] = (
    df_diag_icu['long_title']
    .fillna('')
    .str.lower()
)

In [ ]:
# Create one regular expression containing all keywords
all_keywords = []
for keywords in condition_keywords.values():
    all_keywords.extend(keywords)

pattern = '|'.join(all_keywords)

In [ ]:
# Filter the diagnoses to only include those that match the keywords
df_diag_filtered = df_diag_icu[
    df_diag_icu['long_title'].str.contains(
        pattern,
        regex=True,
        na=False
    )
]

In [ ]:
print(df_diag_filtered.shape)

In [ ]:
# Create the diagnosis flags for each condition based on the filtered diagnoses
for condition, keywords in condition_keywords.items():
    # Create a regex pattern for the keywords of the current condition
    pattern = '|'.join(keywords)

    df_diag_filtered[condition] = (
        df_diag_filtered['long_title']
        .str.contains(pattern, regex=True)
        .astype(int)
    )

In [ ]:
# Keep only the relevant columns for the diagnosis flags
diagnosis_cols = list(condition_keywords.keys())

diagnosis_features = (
    df_diag_filtered[
        ['hadm_id'] + diagnosis_cols
    ]
    .groupby('hadm_id', as_index=False)
    .max()
)

print(diagnosis_features.shape)
diagnosis_features.head()

In [ ]:
# 
diagnosis_features[diagnosis_cols].sum().sort_values(ascending=False)

In [ ]:
# Merge stage 1 features with stage 2 diagnosis features
df_stage2 = df_stage1.merge(
    diagnosis_features,
    on='hadm_id',
    how='left'
)

In [ ]:
# Fill missing values in the diagnosis flags with 0 (indicating absence of the condition)
df_stage2[diagnosis_cols] = (
    df_stage2[diagnosis_cols]
    .fillna(0)
    .astype(int)
)

In [ ]:
print(df_diag_named.shape)
print(df_diag_icu.shape)
print(df_diag_filtered.shape)

In [ ]:
print(df_stage2.shape)

print()

print(df_stage2[diagnosis_cols].isnull().sum())

print()

print(df_stage2.head())

print()

print(df_stage2[diagnosis_cols].sum())

In [ ]:
# save stage 2 features to CSV
df_stage2.to_csv(os.path.join(output_dir, 'icu_stay_features_stage2.csv'), index=False)